In [8]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from torch.optim import AdamW


# Load your dataset
import pandas as pd
data = pd.read_csv("plant_disease_dataset_large.csv")

# Convert to Hugging Face dataset
from datasets import Dataset
dataset = Dataset.from_pandas(data)

# Tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Preprocess function
def preprocess(examples):
    inputs = ["disease: " + cls for cls in examples['disease_class']]
    targets = examples['cure_text']
    model_inputs = tokenizer(inputs, max_length=64, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)

# Training setup
training_args = TrainingArguments(
    output_dir="./t5-disease-model",
    per_device_train_batch_size=4,
    num_train_epochs=10,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    learning_rate=5e-5,
    weight_decay=0.01
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    optimizers=(AdamW(model.parameters(), lr=5e-5), None)  # Use standard AdamW
)

trainer.train()


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss
10,8.436100
20,4.778100
30,2.818500
40,2.136000
50,1.765700
60,1.493100
70,1.131000
80,1.027000
90,0.913900
100,0.843400


TrainOutput(global_step=1250, training_loss=0.3644299861907959, metrics={'train_runtime': 582.7708, 'train_samples_per_second': 8.58, 'train_steps_per_second': 2.145, 'total_flos': 84588625920000.0, 'train_loss': 0.3644299861907959, 'epoch': 10.0})

In [14]:
tokenizer.save_pretrained("./t5-disease-model1")
model.save_pretrained("./t5-disease-model1")


In [16]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("./t5-disease-model1")
model = T5ForConditionalGeneration.from_pretrained("./t5-disease-model1")

input_text = "cure: Pepper__bell___Bacterial_spot"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

outputs = model.generate(input_ids, max_length=300, num_beams=5, early_stopping=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Kur: Pepper__bell___Bacterial_spot
